# 实验一 · Ascend C HelloWorld —— 异构执行模型与核函数

**所属**：《并行计算》第六章 · 昇腾 Ascend C 算子开发　|　**难度**：⭐ 入门　|　**预计时长**：25–35 分钟

第三章至第五章的实验全部运行在 CPU 上：NEON 利用核内的向量部件，Pthreads 与 OpenMP 利用多个 CPU 核心，但程序始终位于同一个地址空间内。从本实验开始，程序被划分到两类处理器上执行，数据不再自动可见，任务不再同步返回。本实验不做任何计算，其目的在于建立并验证这条异构执行链路。

> **实验说明**
> 1. 本实验是第六章的起点，后续全部实验的编译、运行与调试流程均建立在此基础之上。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**，建议在 CANNLab 云开发环境中运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. **源代码通过 `%%writefile` 分段写入同一个 `.asc` 文件**——讲一段、写一段，最后实现 `main` 函数。Host 侧代码与 Device 侧核函数**同处一个文件**，由 `bisheng` 编译器一条命令编译成一个可执行程序。
> 6. 本实验先用 `bisheng` 直接编译单个 `.asc` 文件，随后再演示 CMake 工程方式，并对照二者的适用场景。
> 7. 若环境检查未通过，请先重新运行"环境准备"单元格；仍不通过则在终端执行 `source $ASCEND_TOOLKIT_HOME/set_env.sh` 并重启内核。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明异构计算的主从执行模型，区分 Host 侧与 Device 侧各自的职责
- 说出 AI Core 内部 Scalar / Vector / Cube 三类计算单元与 Local Memory、DMA 的分工，并解释"计算单元只能读 Local Memory"这一约束
- 解释为何 Host 侧指针不能直接传入核函数，以及数据为何必须显式搬运
- 掌握核函数的函数类型限定符 `\_\_global__` 与 `\_\_aicore__` 的作用，以及 `\_\_vector__` / `\_\_cube__` / `\_\_mix__` 三类 Kernel 类型标记的适用场景
- 掌握内核调用符 `<<<blockDim, nullptr, stream>>>` 三个参数的作用与取值范围
- 理解核函数下发的**异步**语义，说明遗漏流同步会导致何种后果
- 掌握 SPMD 执行模型，能通过 `GetBlockIdx()` 与 `GetBlockNum()` 区分不同的计算核
- 用 `bisheng` 编译器把一个 host + device 混写的 `.asc` 文件编译为可执行程序，并说明常用编译选项的含义
- 说明何时才需要引入 CMake 等构建系统

## 🗺️ 学习路径

1. **准备阶段**：理解 Host 与 Device 的职责划分，以及异步下发带来的同步需求
2. **硬件认识**：AI Core 的抽象架构与本机规格，为后续所有"片上占用"核算建立参照
3. **概念建立**：核函数的修饰符、内核调用符的三个参数、SPMD 执行模型
4. **环境检查**：确认 CANN 环境变量、`bisheng` 编译器与 NPU 设备状态
5. **程序实现**：先写 Device 侧核函数，再写 Host 侧主程序，二者同处一个 `.asc` 文件
6. **编译运行**：用 `bisheng` 一条命令编译并执行，观察各核的输出
7. **对照**：改用 CMake 工程方式编译同一份源码，比较两种构建方式的适用场景
8. **结果分析**：讨论输出顺序不确定的成因与 SPMD 对数据切分的意义

## 1. 背景与动机：异构执行模型

在 CPU 并行程序中，线程共享同一个地址空间。主线程分配的数组，工作线程可以直接读写；`pthread_create` 之后，新线程立即开始在同一内存上工作。

异构编程打破了这两条前提。程序被划分为两侧：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 侧别 | 承载硬件 | 职责 |
| --- | --- | --- |
| **Host 侧** | 通用 CPU | 运行管理资源（device / context / stream）申请、内存分配、数据搬运、核函数下发、结果回收 |
| **Device 侧** | 昇腾 AI 处理器 | 执行核函数，完成实际计算 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">侧别</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">承载硬件</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">职责</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>Host 侧</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">通用 CPU</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">运行管理资源（device / context / stream）申请、内存分配、数据搬运、核函数下发、结果回收</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>Device 侧</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">昇腾 AI 处理器</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">执行核函数，完成实际计算</td>
</tr>
</tbody>
</table>

由此产生两项必须由开发者显式处理的事务：

1. **数据不再自动可见。** 两侧拥有各自独立的物理内存。Host 侧的数组必须先经 `aclrtMemcpy` 搬运至 Device 侧，核函数才能访问；计算结果也必须搬回后才能读取。
2. **任务不再同步返回。** 核函数的下发是异步操作，Host 侧下发后立即返回并继续执行后续语句。必须调用 `aclrtSynchronizeStream` 才能确认核函数已执行完毕。

<img src="images/06.01_host_device_arch.png" alt="06.01_host_device_arch" width="780px">

Host 侧是通用服务器，Device 侧是昇腾 AI 处理器，两者通过 PCIe 接口相连。图中 Device 内部的 Global Memory 即设备内存，AI Core 是执行核函数的计算核心。**两侧各有独立的内存空间，这就是数据不再自动可见的物理根源。**

核函数下发之后，设备侧的调度过程如下图所示。

<img src="images/06.01_kernel_schedule.png" alt="06.01_kernel_schedule" width="620px">

这张图同时说明了三件事：Stream 是一个先进先出的任务队列；Host 只负责把 Task 放入队列，放完即返回，这就是异步；队列中的任务由调度单元分配到多个 AI Core 上执行，多核并行由此产生。

**与第四章的对照**：`aclrtSynchronizeStream` 在语义上相当于 `pthread_join` ——二者都是「等待此前发起的异步工作完成」。遗漏它所导致的后果也相似：主流程可能在工作完成之前就释放资源或退出。

**关键差异**：`pthread_join` 等待的是**一个线程**；`aclrtSynchronizeStream` 等待的是 **一条流（Stream）上的整个任务队列**。流是一个 FIFO 队列，同一条流上的任务按下发顺序串行执行——这一点在实验五讲算子链时会成为保证依赖关系的基础。

## 2. 硬件认识：AI Core 内部与本机规格

### 2.1 为什么需要专用硬件

计算两个 16×16 矩阵相乘共需 4096 次乘加。同样的运算量交由三类计算单元完成，所需的运算发射次数相差两个数量级：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 计算单元 | 计算方式 | 所需运算发射次数 |
| --- | --- | --- |
| 标量单元（Scalar） | 三重循环，每次完成一次乘法与一次加法 | `16×16×16×2 = 8192` |
| 矢量单元（Vector） | 每次迭代处理 256 字节数据，即 128 个 float16 元素 | `4096 ÷ 128 = 32` |
| 矩阵单元（Cube） | 每次执行完成两个 16×16 矩阵的乘法 | `1` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">计算单元</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">计算方式</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">所需运算发射次数</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标量单元（Scalar）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三重循环，每次完成一次乘法与一次加法</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>16×16×16×2 = 8192</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矢量单元（Vector）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每次迭代处理 256 字节数据，即 128 个 float16 元素</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>4096 ÷ 128 = 32</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矩阵单元（Cube）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每次执行完成两个 16×16 矩阵的乘法</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>1</code></td>
</tr>
</tbody>
</table>

昇腾 AI Core 内部同时包含这三类计算单元：

- **Cube 矩阵计算单元**：负责执行矩阵运算。以 float16 数据类型为例，每次执行可完成两个 float16 类型的 16×16 矩阵的乘法操作。
- **Vector 矢量计算单元**：负责执行矢量运算。一次迭代（repeat）处理 8 个数据块、共 256 字节，对应 128 个 float16 元素或 64 个 float32 元素。这与第三章的 NEON 属于同一类数据并行——NEON 一条指令处理 4 个 float，矢量计算单元一次迭代处理 64 个。
- **Scalar 标量计算单元**：执行地址计算、循环控制等标量计算工作，并把矢量计算、矩阵计算、数据搬运与同步指令**发射**给对应单元执行。

### 2.2 Ascend C 眼中的 AI Core

真实的 AI Core 结构相当复杂。Ascend C 对它做了一层轻量化抽象，屏蔽不同型号的底层差异，只留下三类组件：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 组件分类 | 组件名称 | 功能 |
| --- | --- | --- |
| 计算单元 | Scalar | 地址计算、循环控制，并把其他指令发射给对应单元 |
| 计算单元 | Vector | 执行矢量运算 |
| 计算单元 | Cube | 执行矩阵运算 |
| 存储单元 | **Local Memory** | AI Core 内部存储（片上），对应数据类型 `LocalTensor` |
| 搬运单元 | **DMA** | 负责 Global Memory ↔ Local Memory 之间的数据搬运 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">组件分类</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">组件名称</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">功能</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">计算单元</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Scalar</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">地址计算、循环控制，并把其他指令发射给对应单元</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">计算单元</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Vector</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">执行矢量运算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">计算单元</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Cube</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">执行矩阵运算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">存储单元</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>Local Memory</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">AI Core 内部存储（片上），对应数据类型 <code>LocalTensor</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">搬运单元</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>DMA</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">负责 Global Memory ↔ Local Memory 之间的数据搬运</td>
</tr>
</tbody>
</table>

<img src="images/06.01_ai_core_abstract_arch.png" alt="06.01_ai_core_abstract_arch" width="780px">

图中三种颜色的箭头对应三类信号流：

- **异步指令流（蓝）**：Scalar 读取指令序列，把矢量、矩阵、搬运指令分别发射到各自的**专属指令队列**；Vector / Cube / DMA **异步并行**执行各自队列里的指令。**这是并行的来源。**
- **同步信号流（绿）**：不同队列的指令之间存在依赖（必须先搬进来才能算），Scalar 下发同步指令来协调时序。
- **计算数据流（红）**：DMA 把数据从 Global Memory 搬到 Local Memory → Vector / Cube 在 Local Memory 上计算 → DMA 把结果搬回 Global Memory。

> **两个必须记住的结论**
> 1. NPU 设备内存（HBM）称为 **Global Memory（GM）**，对应 `GlobalTensor`；AI Core 片上存储称为 **Local Memory**，其中矢量编程用到的部分即 **统一缓冲区（Unified Buffer，UB）**，对应 `LocalTensor`。
> 2. **AI Core 的计算单元只能对 Local Memory 中的数据做计算。** 想算什么，必须先用 DMA 搬进来。
>
> 这与第二章讲的存储层次是同一件事：GM 之于 UB，相当于内存之于 Cache。区别在于，Cache 由硬件自动管理，而 **UB 需要由开发者在代码中显式搬运**。
>
> 需要说明的是，上表是 Ascend C 抽象硬件架构层面的划分。在真实硬件架构中，搬运单元进一步细分为 MTE1、MTE2、MTE3 与 FixPipe，存储单元也细分为 L1 Buffer、L0A/L0B/L0C Buffer、Unified Buffer 等。抽象层屏蔽了这些差异，以降低开发门槛。

下图是 Atlas A2 训练推理系列产品 AI Core 的真实硬件架构，可与上面的抽象架构对照阅读：图中上半部分是矩阵核（AIC），下半部分是矢量核（AIV），二者各有独立的 Scalar 调度单元；抽象架构中统称为 DMA 的搬运单元，在此细分为 MTE1、MTE2、MTE3 与 FixPipe；抽象架构中的 Local Memory，在此细分为 L1 Buffer、L0A / L0B / L0C Buffer 与 Unified Buffer。

<img src="images/06.01_aicore_a2_arch.png" alt="06.01_aicore_a2_arch" width="820px">

本实验只需建立抽象层面的认识。图中各级 Buffer 的分工会在实验六讲矩阵编程时逐一用到。

### 2.3 本机硬件规格（全章的参照表）

从实验二开始，几乎每个版本都要核算方案占用了多少片上缓冲。这些数字只有与硬件容量对照才有意义，因此先给出如下参照表。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 项目 | Atlas A2 / A3 训练与推理系列参考值 | 说明 |
| --- | --- | --- |
| 统一缓冲区 **UB**（矢量编程可用） | **192 KB / 核** | 官方在讨论 bank 冲突时给出该数值 |
| L0C Buffer（Cube 的累加输出） | 128 KB | 官方硬件约束章节给出 |
| L1 Buffer、L0A / L0B Buffer | 该系列的常见配置 | 官方架构规格章节未直接给出容量，要求以接口查询为准 |
| AI Core 构成 | 1 个矩阵核（AIC）配 2 个矢量核（AIV） | 分离模式按 1∶N 组合，本系列为 1∶2 |
| 数据搬运的对齐要求 | **32 字节**（float 即 8 个元素，half 即 16 个元素） | — |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">项目</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">Atlas A2 / A3 训练与推理系列参考值</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">统一缓冲区 <strong>UB</strong>（矢量编程可用）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>192 KB / 核</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">官方在讨论 bank 冲突时给出该数值</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">L0C Buffer（Cube 的累加输出）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">128 KB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">官方硬件约束章节给出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">L1 Buffer、L0A / L0B Buffer</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">该系列的常见配置</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">官方架构规格章节未直接给出容量，要求以接口查询为准</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">AI Core 构成</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1 个矩阵核（AIC）配 2 个矢量核（AIV）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分离模式按 1∶N 组合，本系列为 1∶2</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数据搬运的对齐要求</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>32 字节</strong>（float 即 8 个元素，half 即 16 个元素）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
</tr>
</tbody>
</table>

> ⚠️ 上表数值适用于 Atlas A2 与 A3 训练推理系列，其他产品型号请以官方规格文档为准。官方规格章节并不直接列出各级存储的容量，而是要求在 Host 侧通过 `GetCoreMemSize` 接口查询，在 Device 侧通过 `__NPU_ARCH__` 宏区分架构版本。**本章所有片上占用的核算均以 UB = 192 KB 为分母。**

**为什么 UB 只有 192 KB 这件事很重要**：第二章讲 CPU 缓存时，缓存不命中的后果是**性能下降**；而这里，如果一次要处理的数据放不进 UB，后果是**根本无法计算**，必须先把数据切分成能够放入 UB 的小块。这就是下一个实验要讲的 **Tiling（分块）**。

## 3. 核函数与内核调用符

**核函数（Kernel Function）** 是 Ascend C 算子在设备侧的入口，是 Host 与 Device 之间的桥梁。它看起来就是一个 C++ 函数，只是多了几个函数类型限定符。

<img src="images/06.01_function_call_relation.png" alt="06.01_function_call_relation" width="420px">

图中给出了三类函数的调用关系：host 侧执行函数之间可以相互调用，并通过 `<<<...>>>` 调用核函数；核函数可以调用 device 侧执行函数；device 侧执行函数之间也可以相互调用。**`<<<...>>>` 是两侧之间唯一的调用入口。**

### 3.1 函数类型限定符与 Kernel 类型标记

```cpp
__global__ __vector__ void hello_world()
```

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 限定符 | 类别 | 含义 |
| --- | --- | --- |
| `\_\_global__` | 函数类型限定符 | 标识该函数为**核函数**，可被内核调用符 `<<<>>>` 调用。**必须有** |
| `\_\_aicore__` | 函数类型限定符 | 标识该核函数在设备端 **AI Core** 上执行 |
| `\_\_vector__` | Kernel 类型标记 | 标记为纯矢量类型，核函数在**矢量核（AIV）**上执行 |
| `\_\_cube__` | Kernel 类型标记 | 标记为纯矩阵类型，核函数在**矩阵核（AIC）**上执行 |
| `\_\_mix__(cube, vec)` | Kernel 类型标记 | 标记为混合类型，按给定的 AIC 与 AIV 配比同时在两类核上执行，适用于融合算子 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">限定符</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">类别</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>__global__</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">函数类型限定符</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标识该函数为<strong>核函数</strong>，可被内核调用符 <code>&lt;&lt;&lt;&gt;&gt;&gt;</code> 调用。<strong>必须有</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>__aicore__</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">函数类型限定符</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标识该核函数在设备端 <strong>AI Core</strong> 上执行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>__vector__</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Kernel 类型标记</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标记为纯矢量类型，核函数在<strong>矢量核（AIV）</strong>上执行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>__cube__</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Kernel 类型标记</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标记为纯矩阵类型，核函数在<strong>矩阵核（AIC）</strong>上执行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>__mix__(cube, vec)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Kernel 类型标记</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标记为混合类型，按给定的 AIC 与 AIV 配比同时在两类核上执行，适用于融合算子</td>
</tr>
</tbody>
</table>

> ⚠️ **本实验必须显式标记 `\_\_vector__`。**
>
> 两类限定符各司其职，并非互斥关系：`\_\_global__` 与 `\_\_aicore__` 说明"这是一个在 AI Core 上执行的核函数"，`\_\_vector__` / `\_\_cube__` / `\_\_mix__` 进一步说明"它属于哪种 Kernel 类型"。官方文档中 `\_\_global__ \_\_vector__ \_\_aicore__` 三者连写的示例即由此而来；本实验沿用官方入门教程的简写形式 `\_\_global__ \_\_vector__`。
>
> 编译器可以依据核函数体内的指令自动推导 Kernel 类型，但官方明确列出了三种无法自动推导、必须由开发者手动标记的情形：**纯标量算子**；同一编译单元内存在多个核函数；以及 Atlas 350 加速卡与 Atlas 推理系列产品（无论是否为同一编译单元）。
>
> 本实验的 `hello_world` 函数体内只有一条 `AscendC::printf`，属标量指令，正落在第一种情形上，因此必须显式标记，官方对此推荐的标记即为纯矢量类型。从实验二开始，核函数中会出现 `AscendC::Add` 这类矢量指令，届时可由编译器自动推导。

**关于 `extern "C"`**：本课程的核函数与 `main` 同处一个 `.asc` 文件、由 `<<<>>>` 直接调用，符号在**编译期**就解析完了，不需要 `extern "C"`。只有当核函数被编译成 `.so` 供外部框架按名字查找时才需要它——那是实验七的内容。

### 3.2 返回值与参数规则

- **返回值**：核函数必须是 `void`，不能返回任何值。
- **入参类型**：只支持**指针类型**或 C/C++ 内置基础类型（如 `float*`、`int32_t`）。
- **变量类型限定符**：指针入参必须带 `\_\_gm__`，表明它指向 **Global Memory** 上的地址。Host 侧的指针在 Device 侧没有任何意义，这正是 §1 所述数据不再自动可见在语法层面的体现。
- **宏封装**：框架提供 `GM_ADDR` 宏简化书写：

```cpp
#define GM_ADDR __gm__ uint8_t*
```

所以核函数参数常写成 `GM_ADDR x`，在函数体内再转成实际类型：

```cpp
xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(x), length);
```


### 3.3 内核调用符 `<<<...>>>`

普通 C/C++ 函数调用写作 `function_name(args);`，核函数则使用**内核调用符**额外规定执行配置（官方文档中该参数命名为 `numBlocks`，本实验的源码中命名为 `BLOCK_DIM`）：

```cpp
hello_world<<<blockDim, nullptr, stream>>>();
```

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 参数 | 含义 |
| --- | --- |
| 第一个 `blockDim` | 参与计算的核数，取值范围 **[1, 65535]**，一般设置为物理核数或其倍数。运行时启动该数量的核，各核执行同一份指令代码，各自分配一个逻辑序号 `block_idx`，从 0 开始 |
| 第二个 `l2ctrl` | 保留参数，当前版本固定传入 `nullptr` |
| 第三个 `stream` | 核函数所属的 Stream（任务队列）。同一 Stream 内的任务按下发顺序串行执行 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">参数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第一个 <code>blockDim</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参与计算的核数，取值范围 <strong>[1, 65535]</strong>，一般设置为物理核数或其倍数。运行时启动该数量的核，各核执行同一份指令代码，各自分配一个逻辑序号 <code>block_idx</code>，从 0 开始</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第二个 <code>l2ctrl</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">保留参数，当前版本固定传入 <code>nullptr</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第三个 <code>stream</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核函数所属的 Stream（任务队列）。同一 Stream 内的任务按下发顺序串行执行</td>
</tr>
</tbody>
</table>

> ⚠️ **`<<<>>>` 是异步接口**：它只是把任务提交到 stream 后立即返回，并不等待 Device 侧执行完成。这一点在实验二的性能测量中十分关键——**遗漏同步会得到远高于实际的加速比**，因为此时测得的只是任务下发耗时（几微秒），而非核函数的执行耗时。

### 3.4 Host 侧的固定流程

```text
aclInit → aclrtSetDevice → aclrtCreateStream
        → 【申请内存 + H2D 拷贝】→ 【下发核函数】→ aclrtSynchronizeStream
        → 【D2H 拷贝】→ 释放内存
        → aclrtDestroyStream → aclrtResetDevice → aclFinalize
```

<img src="images/06.01_invocation_steps.png" alt="06.01_invocation_steps" width="260px">

资源的释放顺序与申请顺序**相反**，这与 C 语言中资源管理的一般原则一致。本实验没有真实数据，因此中间带【】的三步暂时省略，实验二会补齐。

需要说明的是，流程中并未出现 `aclrtCreateContext`。`aclrtSetDevice` 会隐式创建一个默认 Context 与一个默认 Stream，单线程程序无需显式管理 Context；多线程或多设备场景下才需要显式创建并绑定。

## 4. SPMD 执行模型

`blockDim` 个核执行**完全相同**的指令代码，唯一的区别是内置变量 `block_idx` 的取值。这就是 SPMD（Single Program Multiple Data）模型。

<img src="images/06.01_multicore_split.png" alt="06.01_multicore_split" width="800px">

*来源：《Ascend C 算子开发指南 01 入门教程》 图 1-3 多核并行处理示意图（该图以 8 个核、总长度 8 × 2048 为例）*

图中把 Global Memory 上一段长度为 `totalLength` 的数据平均分给各核，第 `i` 个核处理的数据起始地址为首地址加上 `blockIdx × blockLength`。各核执行的是同一份指令代码，唯一的差别就是 `blockIdx` 的取值，这正是 SPMD 模型在数据切分上的直接体现。本实验尚未搬运数据，该图描述的是实验二将要实现的形态。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 接口 | 返回值 |
| --- | --- |
| `AscendC::GetBlockIdx()` | 当前核的序号，取值范围 `[0, blockDim)` |
| `AscendC::GetBlockNum()` | 参与计算的核数，即 `blockDim` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">接口</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">返回值</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>AscendC::GetBlockIdx()</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">当前核的序号，取值范围 <code>[0, blockDim)</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>AscendC::GetBlockNum()</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参与计算的核数，即 <code>blockDim</code></td>
</tr>
</tbody>
</table>

**与第五章的对照**：这一模式与 OpenMP 并行区中各线程执行同一段代码、依靠 `omp_get_thread_num()` 区分自身完全一致。

**关键差异**：OpenMP 的各线程共享同一地址空间，可以直接读写同一个数组；而此处各核拥有**彼此独立**的片上缓冲，核之间不存在隐式的数据共享。因此在下一个实验中，多核并行的实现方式必然是「按核序号切分数据、各核处理各自的分片」，而不可能是「多个核同时更新同一块片上数据」。第五章反复讨论的临界区、原子操作与伪共享，在核间这一层次上不会出现。

> **此处先作一点说明，完整讨论留到实验二**：`blockDim` 是逻辑核的概念，它对应的物理资源随 Kernel 类型而变。在分离模式下，纯矢量算子的 `blockDim` 对应启动的矢量核（AIV）数量；纯矩阵算子对应矩阵核（AIC）数量；矢量与矩阵的融合算子则按组合启动，一个组合为 2 个矢量核加 1 个矩阵核，此时 `blockDim` 表示组合数，且不得超过物理核数。本实验已用 `\_\_vector__` 明确了 Kernel 类型，`blockDim` 即参与计算的矢量核数量。实验二将引入 `KERNEL_TASK_TYPE_DEFAULT` 显式声明 Kernel 类型。

## 5. 环境准备与检查

先把 CANN 的环境变量导入 Jupyter 进程，并创建代码目录。**这一步保证 `bisheng` 编译器可用**——Jupyter 内核继承的是启动时的环境，若 CANN 变量是在内核启动后才 source 的，这里必须重新导入一次。

In [ ]:
!mkdir -p src_hello

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")

下面做**环境自检**：确认平台、CANN 版本、`bisheng` 编译器与 NPU 设备状态。如果这一步不通过，后面所有实验都不可能成功，请先解决环境问题。

In [ ]:
import os, platform, shutil, subprocess, sys

print("=" * 62)
print(" 一、平台信息")
print("=" * 62)
print("操作系统   :", platform.system(), platform.release())
print("处理器架构 :", platform.machine())
print("Python    :", sys.version.split()[0])

print()
print("=" * 62)
print(" 二、CANN 与编译器")
print("=" * 62)
ascend_home = os.environ.get("ASCEND_HOME_PATH") or os.environ.get(
    "ASCEND_TOOLKIT_HOME"
)
print("ASCEND_HOME_PATH :", ascend_home or "⚠️  未设置")
bisheng = shutil.which("bisheng")
print("bisheng          :", bisheng or "⚠️  未找到")
if bisheng:
    v = subprocess.run(["bisheng", "--version"], capture_output=True, text=True)
    print(
        "bisheng 版本     :",
        (v.stdout or v.stderr).splitlines()[0] if (v.stdout or v.stderr) else "",
    )
print("npu-smi          :", shutil.which("npu-smi") or "⚠️  未找到")

print()
print("=" * 62)
print(" 三、NPU 设备")
print("=" * 62)
if shutil.which("npu-smi"):
    proc = subprocess.run(["npu-smi", "info"], capture_output=True, text=True)
    print(proc.stdout.rstrip() if proc.returncode == 0 else proc.stderr.rstrip())
else:
    print("未检测到 npu-smi，无法查询设备状态。")

try:
    import acl

    print()
    print("SoC 名称 :", acl.get_soc_name(), "（返回 None 说明运行环境异常）")
except Exception as e:
    print()
    print("acl 模块不可用：", e)

print()
if ascend_home and bisheng:
    print("✅ 环境就绪，可以开始实验。")
else:
    print("⚠️  环境不完整。请重新运行上一个单元格；仍不通过则在终端执行")
    print("    source $ASCEND_TOOLKIT_HOME/set_env.sh")
    print("    随后重启 Notebook 内核。")

## 6. 程序实现：Device 侧核函数与 Host 侧主程序

本实验只有**一个源文件** `src_hello/ascendc_helloworld.asc`，里面同时包含 Device 侧的核函数与 Host 侧的 `main`。

Ascend C 算子的开发流程分为四步，本实验完整地走一遍其中的后三步（环境准备已在 §5 完成）。

<img src="images/06.01_dev_flow.png" alt="06.01_dev_flow" width="300px">

本实验的算子分析极为简单——不做任何计算，只打印核号；重点在后两步：核函数开发（§6.2）与核函数运行验证（§7）。从实验二开始，算子分析将成为独立的一节。

**关于文件后缀**：Ascend C 的源文件使用 `.asc` 后缀，`bisheng` 编译器据此启用异构编译，把同一个文件中的 Host 侧代码与 Device 侧核函数分别编译到对应的目标上，最终链接成**一个**可执行程序。（Device 侧默认支持 C++11 标准，也可指定 C++14、C++17、C++20；受硬件限制，部分 C++ 运行时能力不被支持。）

异构程序的代码天然分为两侧，本节按此顺序组织：**先实现 Device 侧的核函数，再实现 Host 侧的驱动程序**。文件分四段写入，每段前面都有一小节讲解：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 顺序 | 命令 | 归属 | 内容 |
| --- | --- | --- | --- |
| 1 | `%%writefile` | 公共 | 文件头注释与头文件 |
| 2 | `-a` 追加 | **Device 侧** | 核函数 `hello_world` |
| 3 | `-a` 追加 | **Host 侧** | 错误检查宏与核数常量 |
| 4 | `-a` 追加 | **Host 侧** | 主程序 `main` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">顺序</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">命令</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">归属</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>%%writefile</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">公共</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">文件头注释与头文件</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>Device 侧</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核函数 <code>hello_world</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>Host 侧</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">错误检查宏与核数常量</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>Host 侧</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主程序 <code>main</code></td>
</tr>
</tbody>
</table>

> ⚠️ 第一个代码单元格使用 `%%writefile`（**覆盖创建**），其后三个使用 `%%writefile -a`（**追加**）。修改代码后需要从第一个单元格开始按顺序重新执行，否则文件内容会重复或缺失。

### 6.1 源文件组织与头文件

一个 `.asc` 文件需要引入两类头文件：Host 侧应用程序需要的（含 C++ 标准库），以及 Device 侧核函数需要的。二者写在一起，编译器会各自取用。

In [ ]:
%%writefile src_hello/ascendc_helloworld.asc
/**
 * 并行计算 第六章 实验一：Ascend C HelloWorld
 *
 * 本例不做任何计算，其目的在于建立一条最小的可运行链路，用以确认：
 *   （1）CANN 环境变量与 bisheng 编译工具链配置正确；
 *   （2）核函数能够被编译为 Device 侧代码并成功启动；
 *   （3）SPMD 模型下各核依靠 block_idx 区分自身。
 *
 * 本文件同时包含 Host 侧与 Device 侧代码，由一条 bisheng 命令编译。
 */
#include <cstdio>   // Host 侧：std::printf
#include <cstdlib>  // Host 侧：std::exit

#include "acl/acl.h"          // Host 侧：运行管理资源接口
#include "kernel_operator.h"  // Device 侧：Ascend C 编程接口

### 6.2 Device 侧：核函数

核函数是本程序在设备侧执行的部分，其中调用了三个 Ascend C 接口：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 接口 | 作用 |
| --- | --- |
| `AscendC::printf` | 设备侧调试场景下的格式化输出 |
| `AscendC::GetBlockIdx()` | 当前核的逻辑序号，取值 `[0, blockDim)` |
| `AscendC::GetBlockNum()` | 本次任务配置的核数，即 `blockDim` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">接口</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">作用</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>AscendC::printf</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">设备侧调试场景下的格式化输出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>AscendC::GetBlockIdx()</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">当前核的逻辑序号，取值 <code>[0, blockDim)</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>AscendC::GetBlockNum()</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">本次任务配置的核数，即 <code>blockDim</code></td>
</tr>
</tbody>
</table>

按 §3.1 的说明，本核函数体内只有标量指令，属于官方列出的无法自动推导 Kernel 类型的情形，因此必须显式标记 **`\_\_vector__`**。

> ⚠️ **`AscendC::printf` 会影响算子执行性能**：设备侧打印的数据需经缓冲区回传主机侧后才能输出，这一过程会干扰核内的流水执行。**该接口仅用于调试，性能测量时应当移除。** 本章后续实验在测量性能的代码路径上不会出现任何设备侧打印。

In [ ]:
%%writefile -a src_hello/ascendc_helloworld.asc
/* ---------------------------------------------------------------------------
 * Device 侧：核函数
 *   __global__ 表示可由 Host 侧通过内核调用符启动
 *   __vector__ 把 Kernel 类型标记为纯矢量，核函数在矢量核（AIV）上执行
 *     —— 本函数体内只有 printf 这一条标量指令，编译器无法自动推导 Kernel 类型，
 *        因此必须由开发者显式标记。
 *   核函数的返回类型必须为 void。
 * ------------------------------------------------------------------------- */
__global__ __vector__ void hello_world() {
  AscendC::printf("Hello Ascend C! block_idx = %d, block_num = %d\n",
                  static_cast<int32_t>(AscendC::GetBlockIdx()),
                  static_cast<int32_t>(AscendC::GetBlockNum()));
}

### 6.3 Host 侧：错误检查宏与核数常量

Device 侧的代码到此为止，以下均为 Host 侧代码。

先写一个错误检查宏。**每一个 ACL 接口都会返回错误码，都应当检查。** 异构程序的错误往往不会立即导致崩溃，而是在此后很久才以结果全为 0 或程序停止响应的形式表现出来。

一个典型场景是：多人共用一台设备时 `aclrtSetDevice` 可能因设备被占用而失败。若不检查，`stream` 保持为 `nullptr`，后续 `<<<>>>` 与 `aclrtSynchronizeStream` 全部静默失败，程序仍会打印"执行结束"并返回 0：表面上运行成功，实际却没有任何输出，且难以定位原因。

该宏在出错时打印**文件名、行号、出错的表达式与错误码**，随后立即退出。该宏将在本章各实验中反复使用。

同时定义参与计算的核数 `BLOCK_DIM`——它是 Host 侧在 `<<<>>>` 中使用的启动参数，因此归入 Host 侧。

In [ ]:
%%writefile -a src_hello/ascendc_helloworld.asc
/* ---------------------------------------------------------------------------
 * 统一的 ACL 错误检查宏
 *   出错时打印文件、行号、表达式与错误码，随后立即退出。
 *   #expr 是预处理器的"字符串化"运算符，把表达式原文变成字符串。
 * ------------------------------------------------------------------------- */
#define ACL_CHECK(expr)                                                       \
  do {                                                                        \
    aclError _ret = (expr);                                                   \
    if (_ret != ACL_SUCCESS) {                                                \
      std::printf("[ACL ERROR] %s:%d  %s  returned %d\n", __FILE__, __LINE__, \
                  #expr, static_cast<int>(_ret));                             \
      std::exit(EXIT_FAILURE);                                                \
    }                                                                         \
  } while (0)

/* 参与计算的核数。可修改后重新编译，观察输出行数的变化。 */
constexpr uint32_t BLOCK_DIM = 32;

### 6.4 Host 侧：主程序 `main`

`main` 完整地执行一遍 §3.4 的流程，共四个步骤：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 步骤 | 做什么 | 关键点 |
| --- | --- | --- |
| 一 · 申请 | `aclInit` → `aclrtSetDevice` → `aclrtCreateStream` | 准备运行管理资源 |
| 二 · 下发 | `hello_world<<<BLOCK_DIM, nullptr, stream>>>()` | **异步接口**。它只是把任务提交到队列，没有返回值可供检查，错误要到执行时才会显现 |
| 三 · 同步 | `aclrtSynchronizeStream(stream)` | 若删去这一行，Host 侧可能在核函数执行完毕之前就释放资源并退出。**动手练习第 2 题将对此进行验证** |
| 四 · 释放 | `aclrtDestroyStream` → `aclrtResetDevice` → `aclFinalize` | 顺序与申请**相反** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">步骤</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">做什么</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">关键点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一 · 申请</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclInit</code> → <code>aclrtSetDevice</code> → <code>aclrtCreateStream</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">准备运行管理资源</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">二 · 下发</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>hello_world&lt;&lt;&lt;BLOCK_DIM, nullptr, stream&gt;&gt;&gt;()</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>异步接口</strong>。它只是把任务提交到队列，没有返回值可供检查，错误要到执行时才会显现</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三 · 同步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSynchronizeStream(stream)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">若删去这一行，Host 侧可能在核函数执行完毕之前就释放资源并退出。<strong>动手练习第 2 题将对此进行验证</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">四 · 释放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtDestroyStream</code> → <code>aclrtResetDevice</code> → <code>aclFinalize</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">顺序与申请<strong>相反</strong></td>
</tr>
</tbody>
</table>

除 `<<<>>>` 一行外，其余每个 ACL 调用均使用了 `ACL_CHECK`。

至此，整个 `.asc` 文件编写完成。

In [ ]:
%%writefile -a src_hello/ascendc_helloworld.asc
/* ---------------------------------------------------------------------------
 * Host 侧：主程序
 * ------------------------------------------------------------------------- */
int32_t main(int argc, char *argv[]) {
  (void)argc;
  (void)argv;

  int32_t deviceId = 0;
  aclrtStream stream = nullptr;

  /* 步骤一：运行管理资源申请（每一步都检查返回值） */
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(deviceId));
  ACL_CHECK(aclrtCreateStream(&stream));

  /* 步骤二：使用内核调用符 <<<>>> 下发核函数（异步，立即返回）
     *   第一个参数：参与计算的核数 blockDim
     *   第二个参数：保留位，当前固定传入 nullptr
     *   第三个参数：核函数所属的 Stream                              */
  hello_world<<<BLOCK_DIM, nullptr, stream>>>();

  /* 步骤三：核函数的下发是异步的，必须显式同步后才能确认执行完成。
     *         若删去本行，Host 侧可能在核函数执行完毕之前就释放资源并退出。 */
  ACL_CHECK(aclrtSynchronizeStream(stream));

  /* 步骤四：资源释放，顺序与申请顺序相反 */
  ACL_CHECK(aclrtDestroyStream(stream));
  ACL_CHECK(aclrtResetDevice(deviceId));
  ACL_CHECK(aclFinalize());

  std::printf("[实验一] hello_world 执行结束。\n");
  return 0;
}

## 7. 用 bisheng 编译并运行

**毕昇编译器（bisheng）** 是专为昇腾 AI 处理器设计的编译工具，支持异构编程扩展。它同时负责 Host 侧 C++ 代码和 Device 侧 AI Core 指令的编译，因此 **一条命令就能把 host + device 混写的 `.asc` 编译成一个可执行程序**。

### 7.1 常用编译选项

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 编译选项 | 是否必需 | 说明 |
| --- | --- | --- |
| `-help` | 否 | 查看帮助 |
| `--npu-arch=dav-<ver>` | **是** | 编译时指定的 AI 处理器架构版本号，见 7.2 |
| `--npu-soc` | 否 | 编译时指定的 AI 处理器型号；与 `--npu-arch` 同时配置时优先使用 `--npu-arch` |
| `-x` | 否 | 指定编译语言，取值为 `asc` 时表示 Ascend C 编程语言；`.asc` 后缀无需指定 |
| `-o <file>` | 否 | 指定输出文件的名称和位置 |
| `-c` | 否 | 编译生成目标文件 |
| `-shared`、`--shared` | 否 | 编译生成动态链接库 |
| `-lib`、`--cce-build-static-lib` | 否 | 编译生成静态链接库 |
| `-g` | 否 | 编译时增加调试信息 |
| `--sanitizer` | 否 | 编译时增加代码正确性校验信息；需同时添加 `-g`，且不能在 `-O0` 场景下使用 |
| `-fPIC` | 否 | 产生位置无关代码 |
| `-O` | 否 | 指定优化级别，当前支持 `-O3`、`-O2`、`-O0` |
| `--run-mode=sim` | 否 | 仿真模式，链接时使用仿真模式对应的实现库，可查看仿真日志 |
| `-D<宏>=<值>` | 否 | 定义编译期宏。**参数扫描实验依赖该选项，无需修改源码** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">编译选项</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">是否必需</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-help</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">查看帮助</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--npu-arch=dav-&lt;ver&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>是</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译时指定的 AI 处理器架构版本号，见 7.2</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--npu-soc</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译时指定的 AI 处理器型号；与 <code>--npu-arch</code> 同时配置时优先使用 <code>--npu-arch</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-x</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">指定编译语言，取值为 <code>asc</code> 时表示 Ascend C 编程语言；<code>.asc</code> 后缀无需指定</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-o &lt;file&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">指定输出文件的名称和位置</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-c</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译生成目标文件</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-shared</code>、<code>--shared</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译生成动态链接库</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-lib</code>、<code>--cce-build-static-lib</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译生成静态链接库</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-g</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译时增加调试信息</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--sanitizer</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译时增加代码正确性校验信息；需同时添加 <code>-g</code>，且不能在 <code>-O0</code> 场景下使用</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-fPIC</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">产生位置无关代码</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-O</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">指定优化级别，当前支持 <code>-O3</code>、<code>-O2</code>、<code>-O0</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--run-mode=sim</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">仿真模式，链接时使用仿真模式对应的实现库，可查看仿真日志</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-D&lt;宏&gt;=&lt;值&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">定义编译期宏。<strong>参数扫描实验依赖该选项，无需修改源码</strong></td>
</tr>
</tbody>
</table>

### 7.2 `--npu-arch` 取值对照

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| AI 处理器型号 | 架构版本号 |
| --- | --- |
| Atlas 350 加速卡 | `dav-3510` |
| Atlas A3 训练系列产品 / Atlas A3 推理系列产品 | `dav-2201` |
| Atlas A2 训练系列产品 / Atlas A2 推理系列产品 | `dav-2201` |
| Atlas 200I / 500 A2 推理产品 | `dav-3002` |
| Atlas 推理系列产品 | `dav-2002` |
| Atlas 训练系列产品 | `dav-1001` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">AI 处理器型号</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">架构版本号</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Atlas 350 加速卡</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>dav-3510</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Atlas A3 训练系列产品 / Atlas A3 推理系列产品</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>dav-2201</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Atlas A2 训练系列产品 / Atlas A2 推理系列产品</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>dav-2201</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Atlas 200I / 500 A2 推理产品</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>dav-3002</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Atlas 推理系列产品</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>dav-2002</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Atlas 训练系列产品</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>dav-1001</code></td>
</tr>
</tbody>
</table>

> `dav-` 之后的四位数字即 Device 侧预定义宏 `__NPU_ARCH__` 的取值：前三位标识 AI Core 的 IP 核类型，第四位标识同一 IP 核的配置版本。注意 **Atlas A2 与 Atlas A3 共用 2201**，无法通过该宏区分二者。
>
> 请以上一节环境自检中 `npu-smi info` 与 `acl.get_soc_name()` 的实际结果为准选取。若与运行环境不匹配，编译可能通过但运行报错。

In [ ]:
import subprocess

# 追加写入的源文件若被重复执行，会出现内容重复。编译前先做一次自检。
with open("src_hello/ascendc_helloworld.asc", encoding="utf-8") as f:
    _src = f.read()
if _src.count("int32_t main(") != 1:
    raise RuntimeError(
        "源文件中 main 函数出现 %d 次，说明 6.1~6.4 的写入单元格未按顺序执行。"
        "请从 6.1 的 %%%%writefile 单元格开始重新依次执行。" % _src.count("int32_t main(")
    )

ARCH = "dav-2201"  # ← 若设备不是 Atlas A2/A3，请按 7.2 的表修改
SRC = "src_hello/ascendc_helloworld.asc"
EXE = "src_hello/ascendc_helloworld"

# bisheng [算子源文件] --npu-arch=[NPU架构版本号] -O2 -o [输出产物名称]
cmd = ["bisheng", SRC, "--npu-arch=" + ARCH, "-O2", "-o", EXE]
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)

编译成功后执行。预期输出为 BLOCK_DIM 行问候语，`block_idx` 完整覆盖 0 至 BLOCK_DIM-1，其后是主程序打印的一行结束信息。多次运行可以观察到，各行的先后顺序并不固定。

In [ ]:
import subprocess

proc = subprocess.run(
    ["./src_hello/ascendc_helloworld"], capture_output=True, text=True
)
out_npu = proc.stdout
print(out_npu)
if proc.returncode != 0:
    print("返回码:", proc.returncode)
    print(proc.stderr)

## 8. 另一种构建方式：CMake 工程

上一节用一条 `bisheng` 命令完成了编译，简洁直接。当工程规模变大——源文件不止一个、需要链接第三方库、需要按平台或架构条件编译、或者要对外交付算子包时，通常会引入构建系统来管理这些配置。CANN 提供了 CMake 集成方式，下面以**同一份源码**为例演示。

CMake 配置有三个要点：

1. **工具链依赖**：核心是 `find_package(ASC REQUIRED)`，它会自动配置把 `.asc` 编译为 NPU 可执行代码的全部规则。使用前必须设置环境变量 `ASC_DIR` 指向 CANN 的 cmake 模块目录，否则会报 `No CMAKE_ASC_COMPILER could be found`。
2. **双语言支持**：`project(... LANGUAGES ASC CXX)` 让 CMake 分别用正确的编译器处理 Host 侧 C++ 与 Device 侧 Ascend C 代码。
3. **架构指定**：通过 `target_compile_options` 传入 `--npu-arch`，作用与命令行完全相同。

In [ ]:
%%writefile src_hello/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

# find_package(ASC) 用于查找并配置 Ascend C 编译工具链。
# 它依赖环境变量 ASC_DIR，且必须在 project() 之前调用。
find_package(ASC REQUIRED)

# LANGUAGES 中的 ASC 表示使用毕昇编译器编译 Ascend C 代码，
# CXX 用于处理 Host 侧的普通 C++ 代码。
project(ascendc_helloworld LANGUAGES ASC CXX)

add_executable(ascendc_helloworld_cmake ascendc_helloworld.asc)

# 通过编译选项设置 NPU 架构，作用等同于命令行的 --npu-arch
target_compile_options(ascendc_helloworld_cmake PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
)

执行 cmake 构建。

> ⚠️ **Jupyter 中的一个常见问题**：`!` 开头的每一行都是**独立的子 shell**，上一行的 `cd` 和 `export` 对下一行不生效。因此必须用 `&&` 把所有命令串成**一条**，并用反斜杠 `\` 续行。

In [ ]:
!export ASC_DIR=$ASCEND_HOME_PATH/$(uname -m)-linux/tikcpp/ascendc_kernel_cmake/ && \
 cd src_hello && \
 rm -rf build && mkdir -p build && cd build && \
 cmake .. && make && \
 echo "--- 运行 CMake 构建出的可执行文件 ---" && ./ascendc_helloworld_cmake

### 两种方式的对照

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| | `bisheng` 直编 | CMake 工程 |
| --- | --- | --- |
| 命令 | 一行 | 写 `CMakeLists.txt` + `export ASC_DIR` + `cmake` + `make` |
| 交付件 | 1 个 `.asc` | `.asc` + `CMakeLists.txt` + `build/` 目录 |
| 参数扫描 | `-DLAB_BLOCK_DIM=16` 直接追加到命令行 | 需重新配置缓存并重建 |
| 增量编译 | 无（每次全量） | 有依赖跟踪，改一个文件只重编该文件 |
| 适用场景 | 单文件、快速验证、参数扫描 | 多文件、复杂依赖、对外交付算子包 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"></th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"><code>bisheng</code> 直编</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">CMake 工程</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">命令</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一行</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">写 <code>CMakeLists.txt</code> + <code>export ASC_DIR</code> + <code>cmake</code> + <code>make</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">交付件</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1 个 <code>.asc</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>.asc</code> + <code>CMakeLists.txt</code> + <code>build/</code> 目录</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参数扫描</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-DLAB_BLOCK_DIM=16</code> 直接追加到命令行</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">需重新配置缓存并重建</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">增量编译</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">无（每次全量）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">有依赖跟踪，改一个文件只重编该文件</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">适用场景</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单文件、快速验证、参数扫描</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多文件、复杂依赖、对外交付算子包</td>
</tr>
</tbody>
</table>

两种方式产出的可执行程序在功能上等价，输出内容相同；但如 §10 所述，各行的先后顺序并不保证一致。选择哪一种取决于工程的复杂度：源文件只有一个、需要频繁修改参数重复运行时，直接编译更快；源文件较多、依赖较复杂、需要对外交付时，构建系统更便于维护。

## 9. 补充：昇腾的三种调试手段

昇腾官方将算子的调试调优划分为**功能调试**与**性能调优**两类，相关术语是固定的。本课程不展开演示，需要时请查阅 CANN 官方文档的调试调优章节。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 分类 | 子分类 | 是否需要 NPU 设备 | 主要方法 |
| --- | --- | --- | --- |
| **功能调试** | CPU 域孪生调试 | 不需要 | 同一份算子代码在 CPU 域调试精度、NPU 域调试性能；可用 `gdb` 调试并使用 `printf` 打印 |
| **功能调试** | NPU 域上板调试 | 需要 | `printf` 与 `assert` 打印和检查；`DumpTensor` 打印指定 Tensor（仅支持 SIMD 编程场景）；`msDebug` 上板调试工具；`msSanitizer` 内存检测工具 |
| **性能调优** | — | 上板模式需要 | `msprof` 工具采集并分析算子的关键性能指标，支持上板与仿真两种运行模式 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">分类</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">子分类</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">是否需要 NPU 设备</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">主要方法</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>功能调试</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CPU 域孪生调试</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不需要</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同一份算子代码在 CPU 域调试精度、NPU 域调试性能；可用 <code>gdb</code> 调试并使用 <code>printf</code> 打印</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>功能调试</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">NPU 域上板调试</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">需要</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>printf</code> 与 <code>assert</code> 打印和检查；<code>DumpTensor</code> 打印指定 Tensor（仅支持 SIMD 编程场景）；<code>msDebug</code> 上板调试工具；<code>msSanitizer</code> 内存检测工具</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>性能调优</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">上板模式需要</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>msprof</code> 工具采集并分析算子的关键性能指标，支持上板与仿真两种运行模式</td>
</tr>
</tbody>
</table>

**CPU 域孪生调试**中的孪生一词，指的是同一份算子代码既可在 CPU 域调试精度，也可在 NPU 域调试性能。需要说明的是，CPU 域并非指令级仿真器，而是把 Kernel 侧源码用通用的 GCC 编译器编译为 CPU 域二进制，再用 `gdb` 等通用工具调试。它的开启方式与本章的单文件直编不是一条路，需要：

- `#include "tikicpulib.h"` 引入 CPU 调测库
- 用 `AscendC::GmAlloc` / `GmFree` 代替 `aclrtMalloc` / `aclrtFree`
- 用 `ICPU_RUN_KF(kernel, blockDim, ...)` 代替 `<<<blockDim, nullptr, stream>>>`
- 在构建系统里链接 `tikicpulib::${soc_version}`

也就是说，**Host 侧需要另写一套 `main`**。本章统一采用 **NPU 域上板调试**：直接在设备上运行，出现问题时在核函数中临时插入 `AscendC::printf` 打印中间值进行定位。**定位完成后应当删除该语句**，设备侧打印会显著干扰性能测量。

不过这几种手段背后的**方法论**在本章处处适用，请记住：

> **先确认功能正确，再测量性能。未经校验的性能数据没有意义。**

## 10. 结果分析

> 以下结论针对**趋势规律**。具体的输出顺序随运行时调度而变化。

**① 输出行数等于 `blockDim`，与核函数体内是否有循环无关。**

核函数被启动了 BLOCK_DIM 次，每次对应一个核。这与第五章 `#pragma omp parallel` 后并行区被执行 `num_threads` 次的现象同源：**并行的份数由启动参数决定，而非由代码结构决定**。

**② 各行的先后顺序不保证，但 `block_idx` 的取值集合始终完整。**

各核并发执行，而设备侧打印的数据需要先写入缓冲区、回传主机侧后再统一输出，因此屏幕上各行的先后顺序由回传与刷新机制决定。由此引出一条重要的方法论：**不能依据打印顺序推断各核实际的执行先后**，这与第四章中依靠打印交错观察线程调度的做法有本质区别。可以依赖的是另一件事——无论顺序如何变化，`block_idx` 的取值集合始终完整覆盖 `[0, BLOCK_DIM)`，既不重复也不遗漏。这正是 SPMD 模型可用于数据切分的前提。

**③ `blockDim` 是逻辑核的概念，其对应的物理资源随 Kernel 类型而变。**

本实验的 Kernel 类型为纯矢量，`blockDim` 即参与计算的矢量核（AIV）数量，`GetBlockNum()` 的返回值与 `<<<>>>` 中的设置一致。若算子同时包含矢量与矩阵计算，运行时按组合启动，一个组合为 2 个矢量核加 1 个矩阵核，此时 `blockDim` 表示组合数，且不得超过物理核数。实验二将引入 `KERNEL_TASK_TYPE_DEFAULT` 显式声明 Kernel 类型，使核数扫描实验的横轴含义明确无歧义。

本条结论依赖 Kernel 类型与产品型号，属于有条件的结论；①②两条则不依赖具体平台。

**④ 核函数内的打印仅适用于调试。**

打印需要将数据从 Device 侧回传，会干扰核内的流水执行。后续实验在测量性能时均不包含任何打印语句。

---

### 🎓 结论

本实验建立了第六章的两条基本认识：**其一，异构编程中数据不会自动可见、任务不会同步返回，二者都必须显式处理；其二，多核并行的组织方式是 SPMD——各核代码相同、数据不同，核之间没有共享的片上存储。** 后一条决定了实验二中数据切分的基本形态。

## 11. 🔧 动手练习

请修改代码、重新编译并运行，观察结果的变化（建议先独立完成，再继续阅读后续内容）。

> **提示**：修改核函数或 `main` 后需要重新执行 `%%writefile` 单元格。由于第一个单元格为覆盖写、其后三个为追加写，**须从 6.1 开始按顺序重新执行**，否则文件内容会重复或缺失。若仅需修改 `BLOCK_DIM` 这类常量，可直接在编译命令行用 `-D` 覆盖（见练习 1 的提示）。

1. **核数的影响**。将 `BLOCK_DIM` 依次改为 `1`、`4`、`40`，分别重新编译运行，记录输出行数。结合 `npu-smi info` 的输出与 §2.3 的规格表，说明 `blockDim` 的合理取值范围。官方规定其取值范围为 `[1, 65535]`，并建议设置为物理核数或其倍数；本机的矢量核与矩阵核数量可分别通过 `GetCoreNumAiv` 与 `GetCoreNumAic` 接口查询。*提示*：把源码里的 `constexpr uint32_t BLOCK_DIM = 32;` 改成 `#ifndef LAB_BLOCK_DIM` / `#define LAB_BLOCK_DIM 32` / `#endif` + `constexpr uint32_t BLOCK_DIM = LAB_BLOCK_DIM;`，之后就能用 `!bisheng ... -DLAB_BLOCK_DIM=40 -o ...` 一行切换，不必改源码。

2. **异步语义**。注释掉 `ACL_CHECK(aclrtSynchronizeStream(stream));` 一行后重新编译，**连续运行 10 次**，记录输出是否每次都完整。与第四章遗漏 `pthread_join` 的现象作对比。*注意*：这类错误是**非确定性**的，有时正确、有时错误，这正是其危险之处。

3. **条件执行**。在核函数中改为仅当 `GetBlockIdx() == 0` 时才打印，观察输出行数的变化。这一写法对应 OpenMP 中的哪个构造？它与另一个相近的构造有什么区别？

4. **Kernel 类型标记验证**。去掉核函数的 `\_\_vector__` 标记（即只保留 `\_\_global__`，或写成 `\_\_global__ \_\_aicore__`）后重新编译，记录编译器的行为：报错信息、告警，或编译通过但运行结果异常。结合 §3.1 中关于自动推导的三种例外情形解释原因。该验证说明：**编译通过并不等同于实现正确**。

## 12. 🤔 思考题

1. 核函数的参数为何不能传入 Host 侧指针？如果强行传入，程序会在什么阶段出错？

2. `aclrtSynchronizeStream` 与第四章的 `pthread_join` 在语义上有何共同点？二者又有何本质差异？*（提示：一个等待的是线程，另一个等待的是流中的任务队列。如果一条流上先后下发了三个核函数，同步一次等于等待了几个任务？）*

3. BLOCK_DIM 个核的输出顺序每次运行都可能不同，但 `block_idx` 的集合始终完整。为什么可以依赖后者而不能依赖前者？这一性质对数据切分意味着什么？

4. §2.2 说"AI Core 的计算单元只能对 Local Memory 中的数据做计算"。第二章讲的 CPU Cache 也是一层片上存储，为什么 CPU 程序员不需要写"把数据从内存搬到 Cache"的代码，而 Ascend C 程序员必须写？这一差异对编程模型意味着什么？

5. 本实验的 UB 是 192 KB。如果一个算子要处理 8 MB 的 float 数组，在**不做任何分块**的前提下能不能完成？如果不能，应当如何切分？*（该问题将在实验二中系统回答。）*

6. `bisheng` 一条命令就能编译 host + device 混写的文件。那么在什么情况下引入 CMake 才是值得的？请给出至少两个具体判据。

## 13. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| 异构执行模型 | Host 侧负责调度与搬运，Device 侧负责计算，两侧内存相互独立 |
| AI Core 三类计算单元 | Scalar（控制与地址）/ Vector（矢量）/ Cube（矩阵），Scalar 负责发射指令 |
| 存储约束 | **计算单元只能读 Local Memory（UB）**，GM ↔ UB 之间靠 DMA 显式搬运；本系列产品的 UB 为 192 KB，实际容量以 `GetCoreMemSize` 查询为准 |
| 核函数限定符 | `\_\_global__` 与 `\_\_aicore__` 为函数类型限定符；`\_\_vector__` / `\_\_cube__` / `\_\_mix__` 用于标记 Kernel 类型，**纯标量核函数无法自动推导，必须显式标记** |
| 参数规则 | 返回值必须 `void`；指针入参带 `\_\_gm__`，常用 `GM_ADDR` 宏 |
| 内核调用符 | `<<<blockDim, nullptr, stream>>>`，三个参数分别为核数（取值 [1, 65535]）、保留位、流 |
| 异步语义 | 核函数下发后立即返回，须 `aclrtSynchronizeStream` 确认完成 |
| SPMD | 各核共享指令代码，依靠 `block_idx` 区分自身，核间无共享片上存储 |
| 编译 | `bisheng x.asc --npu-arch=dav-2201 -O2 -o demo`，一条命令即可完成 Host 与 Device 代码的编译与链接 |
| 错误处理 | 每个 ACL 调用都应检查返回值；异构程序的错误常常延迟很久才浮现 |
| 调试顺序 | **先确认功能正确，再测量性能。未经校验的性能数据没有意义。** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异构执行模型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Host 侧负责调度与搬运，Device 侧负责计算，两侧内存相互独立</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">AI Core 三类计算单元</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Scalar（控制与地址）/ Vector（矢量）/ Cube（矩阵），Scalar 负责发射指令</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">存储约束</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>计算单元只能读 Local Memory（UB）</strong>，GM ↔ UB 之间靠 DMA 显式搬运；本系列产品的 UB 为 192 KB，实际容量以 <code>GetCoreMemSize</code> 查询为准</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核函数限定符</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>__global__</code> 与 <code>__aicore__</code> 为函数类型限定符；<code>__vector__</code> / <code>__cube__</code> / <code>__mix__</code> 用于标记 Kernel 类型，<strong>纯标量核函数无法自动推导，必须显式标记</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参数规则</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">返回值必须 <code>void</code>；指针入参带 <code>__gm__</code>，常用 <code>GM_ADDR</code> 宏</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">内核调用符</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>&lt;&lt;&lt;blockDim, nullptr, stream&gt;&gt;&gt;</code>，三个参数分别为核数（取值 [1, 65535]）、保留位、流</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步语义</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核函数下发后立即返回，须 <code>aclrtSynchronizeStream</code> 确认完成</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">SPMD</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">各核共享指令代码，依靠 <code>block_idx</code> 区分自身，核间无共享片上存储</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>bisheng x.asc --npu-arch=dav-2201 -O2 -o demo</code>，一条命令即可完成 Host 与 Device 代码的编译与链接</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">错误处理</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个 ACL 调用都应检查返回值；异构程序的错误常常延迟很久才浮现</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">调试顺序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>先确认功能正确，再测量性能。未经校验的性能数据没有意义。</strong></td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **在异构编程中，同构编程下自动完成的事情都不再自动——数据搬运、任务同步、存储层次的管理，都必须由开发者显式写出。**

这既是 Ascend C 编程较为繁琐的原因，也是其性能可控的原因。

### 与后续实验的衔接

➡️ **后续内容：实验二 · 向量加法 Add**。本实验只启动了核函数而未搬运任何数据。下一个实验将回答两个问题：数据如何在 Global Memory 与片上缓冲之间流动，以及 8 个核如何各自处理数据的一部分。届时将引入 Ascend C 最核心的 **三段流水编程范式**，并第一次量化回答：**NPU 到底比 CPU 快多少？**